In [1]:
pip install faiss-cpu sentence-transformers

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
    --------------------------------------- 0.3/18.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/18.9 MB 797.9 kB/s eta 0:00:24
   - -------------------------------------- 0.8/18.9 MB 749.6 kB/s eta 0:00:25
   - -------------------------------------- 0.8/18.9 MB 749.6 kB/s eta 0:00:25
   -- ------------------------------------- 1.0/18.9 MB 686.8 kB/s eta 0:00:26
   -- ------------------------------------- 1.0/18.9 MB 686.8 kB/s eta 0:00:26
   -- ------------------------------------- 1.3/18.9 MB 683.3 kB/s eta 0:00:26
   -- ------------------------------------- 1.3/18.9 MB 683.3 kB/s eta 0:00:26
   --- --------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [13]:
full_df = pd.read_csv('C:\\Users\\Користувач\\Desktop\\NLP_files\\full_df.xls', sep = ';')

In [14]:
full_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2306 entries, 0 to 2305
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   University Name              2306 non-null   object 
 1   Review Text                  2306 non-null   object 
 2   Timestamp                    2306 non-null   object 
 3   Year                         2306 non-null   int64  
 4   Cleaned_Text                 2306 non-null   object 
 5   Lemmatized_Text              2306 non-null   object 
 6   Attitude_Towards_Students    500 non-null    float64
 7   Campus_conditions            500 non-null    float64
 8   Corruption                   500 non-null    float64
 9   Academic_Process_Management  500 non-null    float64
 10  Education_Quality            500 non-null    float64
dtypes: float64(5), int64(1), object(5)
memory usage: 198.3+ KB


In [15]:
full_df.head()

,University Name,Review Text,Timestamp,Year,Cleaned_Text,Lemmatized_Text,Attitude_Towards_Students,Campus_conditions,Corruption,Academic_Process_Management,Education_Quality
0,University_1,1. зміст навчання і предмети не відповідають с...,2024-10-14,2024,1 зміст навчання і предмети не відповідають сп...,NUM зміст навчання предмет не відповідати спец...,NaN,NaN,NaN,NaN,NaN
1,University_1,"Три года обучения дистанционно, хотя в других ...",2023-02-02,2023,три года обучения дистанционно хотя в других в...,три год обучение дистанционно другой вуз студе...,NaN,NaN,NaN,NaN,NaN
2,University_1,"Працюють лише старі як перед смертю, або ті ко...",2023-01-23,2023,працюють лише старі як перед смертю або ті ког...,працювати лише старий смерть немати куди більш...,NaN,NaN,NaN,NaN,NaN
3,University_1,"При поступлении отвратительное отношение, кажд...",2022-09-16,2022,при поступлении отвратительное отношение кажды...,поступление отвратительный отношение считать д...,NaN,NaN,NaN,NaN,NaN
4,University_1,Как бы ректор не заявлял о намерении соответст...,2021-12-15,2021,как бы ректор не заявлял о намерении соответст...,ректор не заявлять намерение соответствовать е...,NaN,NaN,NaN,NaN,NaN


In [37]:

# ==============================
# 1. Завантаження моделі та даних
# ==============================
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

reviews = full_df['Review Text'].tolist()  # список текстів

# ==============================
# 2. Генерація embeddings
# ==============================
embeddings = model.encode(reviews)  # результат np.array (num_reviews x dim)
embeddings = np.array(embeddings).astype('float32')  # FAISS вимагає float32

# ==============================
# 3. Нормалізація для cosine similarity
# ==============================
faiss.normalize_L2(embeddings)

# ==============================
# 4. Створення FAISS індексу (Inner Product)
# ==============================
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # cosine similarity через inner product
index.add(embeddings)

# ==============================
# 5. Пошук схожих відгуків
# ==============================
def search_similar_reviews(query_text, k=5):
    # Генерація embedding для запиту
    query_embedding = model.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_embedding)

    # Пошук топ-k схожих
    D, I = index.search(query_embedding, k)

    # Повертаємо схожі тексти та їх cosine similarity
    similar_reviews = [reviews[i] for i in I[0]]
    similarities = D[0]
    return list(zip(similar_reviews, similarities))


In [40]:
# 6. Приклад використання
# ==============================
query = "Проблеми з поданням документів"
results = search_similar_reviews(query, k=5)

for i, (text, score) in enumerate(results, 1):
    print(f"{i}. Cosine similarity: {score:.4f}")
    print(f"   Review: {text}\n")


1. Cosine similarity: 0.6496
   Review: Организация приема документов неприятно поразила по сравнению с другими ВУЗАми. Все как-то сумбурно, запутанно, что, естественно, породило очереди и неразбериху. Подача документов на две специальности заняла в общем три с половиной часа. Члены приемной комисси нервно призывали подавать документы в электронном виде. Тогда упразните бумажный прием и не путайте детей высказываниями типа "первоочередность за теми, кто привез лично документы".Поразила "обшарпанность" заведения. Руководству университета нужно срочно что-то предпринимать в смысле облагораживания корпусов (хотя бы внутри). Школы в провинции выглядят респектабельней. Охоты поубавилось поступать в этот ВУЗ после посещения.

2. Cosine similarity: 0.5723
   Review: Оставляю отзыв НЕ самому УНИВЕРСИТЕТУ, а Отделению филологии! 

Пару дней назад Я подала туда документы и через несколько дней передумала, потому что меня взяли на другую специальность. Когда я пришла чтобы забрать документы, я ув